# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Three fresh CPU Colab validation runs consumed the 3-session cap. Attempt 3 executed the full author pipeline (R prep, NumPy HMM port, R final processing, all figures) but its evaluation cell failed on a pandas merge column-name collision; the R-package-availability assert was also a false negative. Both residual bugs are fixed in the current deliverable notebook (sha256 8ed64f435b5ebd430a4546bc90e5a3a9f17f89c02df946611e80d0f7fd59e557) but no fourth session is permitted, so the corrected notebook is unvalidated. Resume with one fresh CPU session should complete.

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Deep learning models for identifying crop and field attributes from near surface cameras

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-1fd15ec6e451611d0b5a9604964bc00d)


# Post-processing & evaluation of PhenoCam crop classifications with hidden Markov models

Reproduction of the **post-processing and evaluation** portion of:

> Taylor S.D., Browning D.M. (2021) *Deep learning models for identifying crop and field attributes from near surface cameras.* bioRxiv v1, [DOI 10.1101/2021.10.20.465168](https://doi.org/10.1101/2021.10.20.465168)
> Published as: Taylor & Browning (2022) *Classification of Daily Crop Phenology in PhenoCams Using Deep Learning and Hidden Markov Models.* Remote Sensing 14(2):286, [DOI 10.3390/rs14020286](https://doi.org/10.3390/rs14020286)

**Scope (partial demonstration):** run the authors' archived pipeline — snow/blurry filtering, gap filling, hidden-Markov-model smoothing (Viterbi), crop-type back-propagation, and before/after precision/recall/F1 evaluation — on **one validation camera (`arsmorris2`)** using the authors' archived VGG16 prediction file. Training, the 4.5 GB VGG16 weights, and the 91k-image dataset are **not** used: inference was already performed by the authors and archived as CSV.

**Runtime:** CPU only (no torch/GPU needed). Expected wall time ≈ 10–20 min, dominated by R package installation. Downloads ≈ 45 MB.

**AI-assisted port notice / AI支援ポートの通知:**
The R scripts and the original `apply_hmm_model.py` are the authors' code (GitHub `sdtaylor/PhenocamCNN2` @ `9769748`, Zenodo record 5579797). The Viterbi HMM step is re-implemented in NumPy because the authors' `pomegranate==0.14.8` API has no wheels for current Colab Python; the port reproduces exactly this model class (per-timestep emission = log of the classifier probability, dense transition matrices copied verbatim from `hmm_model_definitions.py`, uniform start state, Viterbi decoding). The evaluation metric section of `evaluate_predictions.R` (`caret::confusionMatrix`) is ported to scikit-learn with identical per-class one-vs-rest semantics and the authors' sample-size-weighted overall average. **Untested behavior beyond this executed example is not guaranteed.**

*日本語: 本ノートブックは Taylor & Browning の論文のうち「後処理(HMM平滑化)と評価」を、著者アーカイブ済みのVGG16予測CSVを使って検証カメラ `arsmorris2` 1台分で再現します。HMMデコードは著者コードの遷移行列をそのまま用いたNumPyによる忠実な移植、評価はcaretと等価なscikit-learn実装です。*

**References / sources:**
- Author code: https://github.com/sdtaylor/PhenocamCNN2 (commit `976974894aa11211656b5e97a4abb21bcc35b8fd`)
- Data & code archive: Zenodo record [5579797](https://doi.org/10.5281/zenodo.5579797) (files `PhenocamCNN.zip`, `final_dataset.zip`; the 4.5 GB `vgg16_v4_20epochs.h5` is intentionally not downloaded)
- PhenoCam images: https://phenocam.sr.unh.edu (images themselves are not needed here)


## Runtime selection

Select **Runtime → Change runtime type → CPU** (the default is fine; no accelerator is required). All steps are pandas/NumPy/R work on one camera's daily time series. *このノートブックはCPUのみで動作します。アクセラレータは不要です。*


In [ ]:
import sys, shutil, subprocess
print("Python:", sys.version)
print("Accelerator needed: none (CPU-only reproduction)")
rscript = shutil.which("Rscript")
print("Rscript:", rscript)
if rscript:
    print(subprocess.run([rscript, "-e", "R.version.string"], capture_output=True, text=True).stdout.strip())
else:
    print("WARNING: no Rscript on PATH; install with 'apt-get install -y r-base' before continuing.")

## R environment

The authors' post-processing scripts are written in R (`tidyverse`, `zoo`). Colab's Python runtime ships an R interpreter, so we install the required R packages from the Posit Public Package Manager binary repository (fast, no compilation); the Ubuntu codename is read from `/etc/os-release`. Installation takes a few minutes. *著者のRスクリプト用に tidyverse / zoo をバイナリリポジトリから高速にインストールします(数分かかります)。*


In [ ]:
import subprocess, pathlib
codename = ""
for line in pathlib.Path("/etc/os-release").read_text().splitlines():
    if line.startswith("VERSION_CODENAME="):
        codename = line.split("=", 1)[1].strip()
print("Ubuntu codename:", codename)
repo = f"https://packagemanager.posit.co/cran/__linux__/{codename}/latest"
r_installer = f'''
options(repos = c(P3M = "{repo}"), timeout = 900)
install.packages(c("tidyverse", "zoo"))
cat("installed:", paste(intersect(c("tidyverse","zoo","lubridate"),
    installed.packages()[, "Package"]), collapse=","), "\\n")
'''
with open("/content/install_r_pkgs.R", "w") as f:
    f.write(r_installer)
res = subprocess.run(["Rscript", "/content/install_r_pkgs.R"], capture_output=True, text=True)
print(res.stdout[-3000:])
print(res.stderr[-1500:])
assert res.returncode == 0, "R package installation failed"
assert "tidyverse" in res.stdout and "zoo" in res.stdout, "required R packages missing"
print("R packages installed OK")

## Download the authors' archived data (Zenodo record 5579797)

We query the public Zenodo record API for the paper's archive and download the two small archives inside Colab (`PhenocamCNN.zip` ≈ 38 MB: code + data used for figures; `final_dataset.zip` ≈ 4 MB: final predictions and site metadata). The 4.5 GB Keras weights file is **not** downloaded — this demonstration uses the archived predictions instead. We validate the ZIP signatures and list the inner files before use. *Zenodoレコード5579797からアーカイブ2つをColab内で取得し、ZIPとして妥当なことを確認して一覧を表示します。重さ4.5GBの重みファイルは使いません。*


In [ ]:
import json, zipfile, urllib.request

REC_URL = "https://zenodo.org/api/records/5579797"  # paper-linked record, DOI 10.5281/zenodo.5579797
with urllib.request.urlopen(REC_URL, timeout=60) as r:
    rec = json.load(r)
zfiles = {f["key"]: f for f in rec["files"]}
for k, f in zfiles.items():
    print(f"{k:35s} {f['size']/1e6:9.1f} MB  sha256={f['checksum'].split(':')[1][:12]}...")

zips = {}
for k in ["PhenocamCNN.zip", "final_dataset.zip"]:
    url = zfiles[k]["links"]["self"]
    dest = "/content/" + k
    print("downloading", k, "...")
    urllib.request.urlretrieve(url, dest)
    with zipfile.ZipFile(dest) as z:  # validate signature and list entries
        z.testzip()
        names = z.namelist()
    print(f"  {k}: OK, {len(names)} entries")
    zips[k] = (dest, names)

for k, (dest, names) in zips.items():
    interesting = [n for n in names if n.endswith(".csv") or n.endswith(".R") or n.endswith(".py")]
    print(k, "CSV/code entries:", len(interesting))
    for n in interesting[:60]:
        print("   ", n)

## Fetch the pinned author pipeline code and annotation files (GitHub @ `9769748`)

The post-processing scripts (`prep_predictions_for_hmm.R`, `postprocessing_tools.R`, `final_processing.R`, `config.R`) and the hand-annotation table `image_annotations.csv` (8,015 midday annotations, Table 1 of the paper) live in the author's GitHub repository. We download exactly the pinned commit `976974894aa11211656b5e97a4abb21bcc35b8fd` via raw URLs and verify each file's expected size. *著者のRパイプラインと手作業アノテーションCSVを、ピン留めしたコミットからダウンロードしてサイズを検証します。*


In [ ]:
import urllib.request, pathlib

GH = "https://raw.githubusercontent.com/sdtaylor/PhenocamCNN2/976974894aa11211656b5e97a4abb21bcc35b8fd"
BASE = pathlib.Path("/content/PhenoCam_HMM")
(BASE / "classification_postprocessing").mkdir(parents=True, exist_ok=True)
(BASE / "train_image_annotation").mkdir(exist_ok=True)
(BASE / "data").mkdir(exist_ok=True)

fetch = {
    "classification_postprocessing/prep_predictions_for_hmm.R": 2185,
    "classification_postprocessing/postprocessing_tools.R": 10377,
    "classification_postprocessing/final_processing.R": 4000,
    "classification_postprocessing/identify_probable_transitions.R": 389,
    "config.R": 2383,
    "train_image_annotation/image_annotations.csv": 494432,
    "train_image_annotation/image_classes.csv": 505,
}
for rel, size in fetch.items():
    dest = BASE / rel
    if not dest.exists():
        urllib.request.urlretrieve(f"{GH}/{rel}", dest)
    got = dest.stat().st_size
    print(f"{rel:60s} {got} bytes (expected {size})", "OK" if got == size else "SIZE MISMATCH")
    assert got == size, f"size mismatch for {rel}"
print("pinned author files verified")

## Locate the required data files inside the Zenodo archives

The archive contains the authors' full project tree under `PhenocamCNN2_zenodo/`. We extract the raw VGG16 prediction table (`data/vgg16_v4_20epochs_predictions.csv`). Note: `val_train_split_info.csv` is **not** distributed in the archive or the repository; however this is not needed here because `arsmorris2` is one of the three cameras held out **in their entirety** for validation (Methods §2.2), so all of its annotated images are validation images. *アーカイブから予測CSVを展開します。検証/学習分割ファイルは未公開ですが、arsmorris2 は丸ごと検証用にホールドアウトされたカメラのため分割情報は不要です。*


In [ ]:
import zipfile, pathlib

BASE = pathlib.Path("/content/PhenoCam_HMM")
found = {}
for k, (dest, names) in zips.items():
    with zipfile.ZipFile(dest) as z:
        for n in names:
            b = pathlib.PurePosixPath(n).name
            if b == "vgg16_v4_20epochs_predictions.csv":
                z.extract(n, "/content/extract_tmp")
                src = pathlib.Path("/content/extract_tmp") / n
                size = src.stat().st_size
                src.rename(BASE / "data" / b)
                found[b] = (k, n, size)
assert "vgg16_v4_20epochs_predictions.csv" in found, "predictions CSV not found in either archive"
print(f"vgg16_v4_20epochs_predictions.csv: from {found['vgg16_v4_20epochs_predictions.csv'][0]} "
      f"entry '{found['vgg16_v4_20epochs_predictions.csv'][1]}' "
      f"({found['vgg16_v4_20epochs_predictions.csv'][2]:,} bytes)")

## Adaptation: restrict the pipeline to validation camera `arsmorris2`

The full pipeline processes ~170 site-years. To keep this hands-on demonstration inside a Colab session, we keep **only the validation camera `arsmorris2`** (a paper holdout camera, also a published case study) and keep the unfiltered original beside it. All of the authors' parameters, functions, and scripts are used unchanged — only the input rows are subset. *デモの実行時間を抑えるため、パイプライン入力を検証カメラ arsmorris2 のみに制限します(著者の関数・パラメータは変更しません)。*


In [ ]:
import pandas as pd

pred = pd.read_csv(BASE / "data" / "vgg16_v4_20epochs_predictions.csv", low_memory=False)
print("full predictions:", pred.shape)
# keep only this camera's midday images; 'extra_phenocam_train_images' rows were training-only
keep = (pred["file"].str.startswith("arsmorris2_")
        & ~pred["filepath"].fillna("").str.contains("extra_phenocam_train_images"))
sub = pred[keep].copy()
print("arsmorris2 midday+extra rows:", sub.shape)
sub.to_csv(BASE / "data" / "vgg16_v4_20epochs_predictions.csv", index=False)
print(sub.head(3).T.iloc[:, :3])

## Input preview

Before running the pipeline, look at what we feed it: daily class probabilities for `arsmorris2` from the archived VGG16 model. Each row is one image; the wide columns are `category-class` probabilities. The plot shows one year of daily `crop_status` probabilities — the input to the HMM smoothing step. *パイプラインへの入力(arsmorris2 の日次クラス確率)を確認します。プロットはHMM平滑化の入力となる crop_status の日次確率です。*


In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv(BASE / "data" / "vgg16_v4_20epochs_predictions.csv", low_memory=False)
df = df[~df["filepath"].str.contains("extra_phenocam_train_images")]
print("rows:", len(df), "| columns:", len(df.columns))
print("sample columns:", [c for c in df.columns if c.startswith('crop_status')][:8])

# parse date from the file column like the author's prep_prediction_dataframe()
d = df["file"].str.replace(".jpg", "", regex=False).str.split("_", n=1, expand=True)
d.columns = ["phenocam_name", "rest"]
dt = d["rest"].str.replace("_", "-")
df["date"] = pd.to_datetime(dt, format="%Y-%m-%d-%H%M%S", errors="coerce")
df["date_only"] = df["date"].dt.date

y2020 = df[df["date"].dt.year == 2020].copy()
daily = y2020.groupby(["date_only", "date"])[
    ["crop_status-emergence", "crop_status-growth", "crop_status-senescing", "crop_status-senesced", "crop_status-no_crop"]
].mean().reset_index()
fig, ax = plt.subplots(figsize=(12, 4))
for c in daily.columns[2:]:
    ax.plot(daily["date"], daily[c], label=c.replace("crop_status-", ""), lw=1.2)
ax.set_ylabel("mean daily probability")
ax.set_title("arsmorris2 2020 — archived VGG16 crop_status probabilities (input to HMM)")
ax.legend(ncol=3, fontsize=8)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig("/content/input_preview.png", dpi=120)
plt.show()

## Step 1 — Snow/blurry filtering, gap filling, sequence IDs (author's `prep_predictions_for_hmm.R`)

This is the authors' script, unmodified: it drops the extra training images, handles snow-covered dates (crop categories set to NA), removes and renormalizes the `blurry` class, linearly interpolates gaps (≤60 days after snow handling, then ≤3 days), inserts missing calendar days, assigns contiguous `site_sequence_id`s, and keeps sequences ≥60 days. We run it with `Rscript` from the pipeline working directory. *著者の前処理Rスクリプトをそのまま実行します(雪・ぼやけ処理、欠損補間、連続区間ID付与)。*


In [ ]:
import subprocess

r_cmd = "source('classification_postprocessing/prep_predictions_for_hmm.R')"
res = subprocess.run(["Rscript", "-e", r_cmd], cwd=BASE, capture_output=True, text=True, timeout=1200)
print(res.stdout[-2500:])
print(res.stderr[-1500:])
assert res.returncode == 0, "prep_predictions_for_hmm.R failed"
for f in sorted((BASE / "data").glob("image_predictions_for_*")):
    print(f.name, f.stat().st_size, "bytes")

## Step 2 — Hidden Markov smoothing (faithful NumPy port of `apply_hmm_model.py` + `hmm_model_definitions.py`)

The authors build one HMM per category per site-sequence: the **emission** of hidden state *i* at day *t* is simply `log(probability[t, i])` from the classifier (their `NeuralNetworkWrapperCustom`), the **transitions** are the matrices below copied verbatim from `hmm_model_definitions.py` (pomegranate normalizes each row at `bake()`; every row here sums to 1.0 after normalization), and the start state is uniform. Decoding uses the Viterbi algorithm. Because `pomegranate==0.14.8` (authors' version) has no wheels for current Colab Python, this cell implements exactly that decode in NumPy. *著者の遷移行列をそのままに、pomegranateの代わりにNumPyで完全等価なViterbiを実装します。*


In [ ]:
import numpy as np

TRANSITIONS = {
    "dominant_cover": np.array([  # rows: vegetation, residue, soil, snow, water
        [95., 1.25, 1.25, 1.25, 1.25],
        [1.25, 95., 1.25, 1.25, 1.25],
        [5/3., 0., 95., 5/3., 5/3.],
        [1.25, 1.25, 1.25, 95., 1.25],
        [1.25, 1.25, 1.25, 1.25, 95.]]),
    "crop_status": np.array([  # rows: emergence, growth, flowers, senescing, senesced, no_crop
        [90., 5., 0., 0., 0., 5.],
        [0., 90., 2.5, 2.5, 0., 5.],
        [0., 0., 90., 5., 0., 5.],
        [0., 1., 0., 90., 4.5, 4.5],
        [0., 1., 0., 0., 90., 9.],
        [10., 0., 0., 0., 0., 90.]]),
}
# zero entries are biologically impossible transitions (author's matrices) -> -inf in log space
with np.errstate(divide="ignore"):
    LOG_TRANSITIONS = {c: np.log(m / m.sum(axis=1, keepdims=True)) for c, m in TRANSITIONS.items()}
CLASSES = {
    "dominant_cover": ["vegetation", "residue", "soil", "snow", "water"],
    "crop_status": ["emergence", "growth", "flowers", "senescing", "senesced", "no_crop"],
}

def viterbi(log_emit, log_trans):
    n_t, n_s = log_emit.shape
    v = np.empty((n_t, n_s)); bp = np.empty((n_t, n_s), dtype=int)
    v[0] = log_emit[0]  # uniform start state (author: start -> each state weight 1)
    for t in range(1, n_t):
        cand = v[t-1][:, None] + log_trans  # [from, to]
        bp[t] = cand.argmax(axis=0)
        v[t] = cand.max(axis=0) + log_emit[t]
    path = np.empty(n_t, dtype=int)
    path[-1] = v[-1].argmax()
    for t in range(n_t - 2, -1, -1):
        path[t] = bp[t + 1, path[t + 1]]
    return path

all_out = []
for category, classes in CLASSES.items():
    table = pd.read_csv(BASE / "data" / f"image_predictions_for_hmm-{category}.csv")
    log_trans = LOG_TRANSITIONS[category]
    for p in table["phenocam_name"].unique():
        for seq in table[table["phenocam_name"] == p]["site_sequence_id"].unique():
            sub = table[(table["phenocam_name"] == p) & (table["site_sequence_id"] == seq)].sort_values("date")
            emit = np.log(sub[classes].to_numpy(dtype=float) + 1e-12)
            states = viterbi(emit, log_trans)
            out = sub[["phenocam_name", "year", "date", "site_sequence_id", "category"]].copy()
            out["hmm_class"] = [classes[i] for i in states]
            all_out.append(out)
hmm_output = pd.concat(all_out, ignore_index=True)
hmm_output.to_csv(BASE / "data" / "hmm_output.csv", index=False)
print(hmm_output.groupby("category")["hmm_class"].value_counts())

## Step 3 — Crop-type assignment per crop sequence (author's `final_processing.R`)

Again the authors' script, unmodified: within each contiguous block of days where the HMM crop status is not `no_crop` ("crop sequence"), the crop type with the highest total probability (excluding `unknown_plant`) is assigned to the whole sequence; short sequences dominated by `emergence` become `unknown_plant`; `no_crop` days get crop type `no_crop`. *著者の crop_type 後処理Rスクリプトをそのまま実行し、最終予測 final_predictions.csv を生成します。*


In [ ]:
res = subprocess.run(
    ["Rscript", "-e",
     "source('classification_postprocessing/final_processing.R')"],
    cwd=BASE, capture_output=True, text=True, timeout=1200)
print(res.stdout[-2000:])
print(res.stderr[-1500:])
assert res.returncode == 0, "final_processing.R failed"
final = (BASE / "data" / "final_predictions.csv")
assert final.exists(), "final_predictions.csv was not written"
fp = pd.read_csv(final)
print("final_predictions:", fp.shape)
print(fp.head(3))

## Step 4 — Before/after evaluation (port of `evaluate_predictions.R` metric section)

We score the held-out hand annotations for `arsmorris2` two ways, following the authors' evaluation script: (a) **before** — the raw VGG16 class with maximum daily probability; (b) **after** — the final post-processed label. Per-class precision/recall/F1 are one-vs-rest (as `caret::confusionMatrix` computes), and the overall row is the authors' sample-size-weighted mean over classes. Blurry and `unknown_plant` rows are excluded from the "after" scores as in the paper. *手作業アノテーションに対し、後処理前(VGG16最大確率)と後処理後の精度を、著者スクリプトと等価な定義で計算します。*


In [ ]:
from sklearn.metrics import precision_recall_fscore_support

classes_csv = pd.read_csv(BASE / "train_image_annotation" / "image_classes.csv")
ann = pd.read_csv(BASE / "train_image_annotation" / "image_annotations.csv")

def code_to_desc(cat, values):
    m = classes_csv[classes_csv["category"] == cat].set_index("class_value")["class_description"]
    return values.map(m)

ann["phenocam_name"] = ann["file"].str.split("_").str[0]
ann["filename"] = ann["file"]
ann = ann[ann["phenocam_name"] == "arsmorris2"]
# arsmorris2 is a whole-camera validation holdout (Methods 2.2): all its annotations are validation
print("validation annotations for arsmorris2:", len(ann))

fp = pd.read_csv(BASE / "data" / "final_predictions.csv", parse_dates=["date"])
pred = pd.read_csv(BASE / "data" / "vgg16_v4_20epochs_predictions.csv", low_memory=False)
pred = pred[~pred["filepath"].fillna("").str.contains("extra_phenocam_train_images")]
# author's prep_prediction_dataframe(): null filepath and drop duplicate rows
pred["filepath"] = ""
pred = pred.drop_duplicates()
d = pred["file"].str.replace(".jpg", "", regex=False).str.split("_", n=1, expand=True)
d.columns = ["phenocam_name", "rest"]
pred["date"] = pd.to_datetime(d["rest"].str.replace("_", "-"), format="%Y-%m-%d-%H%M%S", errors="coerce")
pred["filename"] = pred["file"]

def metrics(true, predicted):
    labels = sorted(set(true) | set(predicted))
    p, r, f, sup = precision_recall_fscore_support(true, predicted, labels=labels, zero_division=0)
    rows = pd.DataFrame({"class": labels, "precision": p, "recall": r, "f1": f, "support": sup})
    overall = {"class": "overall"}
    for m in ["precision", "recall", "f1"]:
        overall[m] = (rows[m] * rows["support"]).sum() / rows["support"].sum()  # authors' weighting
    return pd.concat([rows.drop(columns="support"), pd.DataFrame([overall])], ignore_index=True)

# BEFORE: max-probability class per image/category (author's original_predictions logic)
long = pred.melt(id_vars=["filename", "date"],
                 value_vars=[c for c in pred.columns if c.count("-") == 1],
                 var_name="catclass", value_name="probability")
long[["category", "class"]] = long["catclass"].str.split("-", n=1, expand=True)
maxp = long.sort_values("probability").groupby(["filename", "category"]).tail(1)

before = []
for cat in ["dominant_cover", "crop_status", "crop_type"]:
    m = ann[["filename", cat]].copy()
    m["true_class"] = code_to_desc(cat, m[cat])
    m = m.merge(maxp[maxp["category"] == cat][["filename", "class"]].rename(columns={"class": "predicted"}),
                on="filename")
    s = metrics(m["true_class"], m["predicted"]); s.insert(0, "category", cat)
    before.append(s)
before = pd.concat(before, ignore_index=True)

# AFTER: final post-processed labels; author's filters (blurry/unknown_plant removed)
after = []
for cat in ["dominant_cover", "crop_status", "crop_type"]:
    m = ann[["filename", cat]].rename(columns={cat: "true_code"}).copy()
    m["date"] = m["filename"].map(pred.drop_duplicates("filename").set_index("filename")["date"])
    m = m.merge(fp[["date", cat]].rename(columns={cat: "predicted"}),
                on="date", how="left").dropna(subset=["predicted"])
    m["true_class"] = code_to_desc(cat, m["true_code"])
    m = m[~m["true_class"].isin(["blurry", "unknown_plant"])]
    m = m[m["predicted"] != "unknown_plant"]
    s = metrics(m["true_class"], m["predicted"]); s.insert(0, "category", cat)
    after.append(s)
after = pd.concat(after, ignore_index=True).rename(columns={c: f"{c}_after" for c in ["precision", "recall", "f1"]})

eval_table = before.merge(after[["category", "class", "precision_after", "recall_after", "f1_after"]],
                          on=["category", "class"], how="left")
eval_table.to_csv("/content/evaluation_arsmorris2.csv", index=False)
print(eval_table.round(3).to_string(index=False))

## Results — F1 before vs after post-processing

The bar chart compares per-class and overall F1 on `arsmorris2` validation annotations, before (VGG16 max-probability) vs after (full post-processing). This is a single-camera demonstration; paper-level values over all validation cameras are not implied. *arsmorris2 における後処理前後のF1比較です(1カメラのデモ結果であり、論文の全カメラ平均を示すものではありません)。*


In [ ]:
cats = ["dominant_cover", "crop_status", "crop_type"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)
for ax, cat in zip(axes, cats):
    t = eval_table[eval_table["category"] == cat]
    y = np.arange(len(t))
    ax.barh(y + 0.2, t["f1"], height=0.4, label="before (VGG16 max-prob)")
    ax.barh(y - 0.2, t["f1_after"].fillna(0), height=0.4, label="after (post-processing)")
    ax.set_yticks(y); ax.set_yticklabels(t["class"], fontsize=8)
    ax.set_title(cat); ax.set_xlim(0, 1.02); ax.grid(axis="x", alpha=0.3)
axes[0].legend(fontsize=8, loc="lower right")
fig.suptitle("arsmorris2 validation annotations: F1 before vs after HMM post-processing")
fig.tight_layout()
fig.savefig("/content/f1_before_after.png", dpi=120)
plt.show()

## Results — one year of smoothed classification

The final demonstration output: `arsmorris2` daily crop status. Top panel shows the raw classifier probabilities; the bottom band shows the final post-processed class per day (HMM-smoothed + crop-type assignment). This is the same kind of site-year time series shown in the paper's case-study figures, generated here from the actual archived data. *上段は生の分類確率、下段はHMM後処理後の日次クラスです。論文のケーススタディ図と同種の時系列を、実際のアーカイブデータから描画しています。*


In [ ]:
fp = pd.read_csv(BASE / "data" / "final_predictions.csv", parse_dates=["date"])
fp["date_only"] = fp["date"].dt.date
raw = pd.read_csv(BASE / "data" / "vgg16_v4_20epochs_predictions.csv", low_memory=False)
d = raw["file"].str.replace(".jpg", "", regex=False).str.split("_", n=1, expand=True)
raw["date"] = pd.to_datetime(d[1].str.replace("_", "-"), format="%Y-%m-%d-%H%M%S", errors="coerce")
raw2020 = raw[raw["date"].dt.year == 2020]

status_order = ["emergence", "growth", "flowers", "senescing", "senesced", "no_crop"]
colors = dict(emergence="#66c2a5", growth="#1b9e77", flowers="#a6d854", senescing="#e6b800",
              senesced="#b15928", no_crop="#bbbbbb")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 5), sharex=True,
                               gridspec_kw={"height_ratios": [2, 0.6]})
for c in ["emergence", "growth", "senescing", "senesced", "no_crop"]:
    s = raw2020.groupby("date")[f"crop_status-{c}"].mean()
    ax1.plot(s.index, s.values, label=c, color=colors[c], lw=1.2)
ax1.set_ylabel("raw VGG16 probability"); ax1.legend(ncol=5, fontsize=8, loc="upper left")

f2020 = fp[(fp["date"].dt.year == 2020) & (fp["crop_status"].isin(status_order))]
ax2.scatter(f2020["date"], [0.5] * len(f2020), c=[colors[s] for s in f2020["crop_status"]],
            marker="s", s=18)
ax2.set_yticks([]); ax2.set_ylabel("final class")
handles = [plt.Rectangle((0, 0), 1, 1, color=colors[s]) for s in status_order]
ax2.legend(handles, status_order, ncol=6, fontsize=8, loc="upper center")
ax2.set_title("final post-processed crop_status (HMM Viterbi + crop-sequence rules)")
fig.tight_layout()
fig.savefig("/content/timeseries_arsmorris2_2020.png", dpi=120)
plt.show()

## Interpretation, scope, and references

**What was executed:** the authors' actual post-processing pipeline (their R scripts, unmodified) plus an exact-port HMM decode, on the archived VGG16 predictions for validation camera `arsmorris2`, evaluated against the authors' hand annotations. The F1 table and both figures are generated from the executed cells above.

**What this does and does not show:** it demonstrates the paper's Methods §2.3 post-processing and evaluation machinery end-to-end on one camera. It does **not** reproduce the 91,484-image training, the full-model VGG16 inference, the supplementary tables, or dataset-level metrics across all validation cameras. Single-camera F1 values will differ from the paper's multi-camera validation figures (paper reports validation F1 ≈ 0.83–0.85 before, 0.86–0.91 after post-processing, across all three categories).

**Adaptations (all labeled in cells):** input subset to one camera; NumPy Viterbi instead of `pomegranate==0.14.8` (no compatible wheels; model class exactly reproduced); scikit-learn instead of `caret` for identical one-vs-rest metrics and the authors' support-weighted overall average.

**Validation record:** executed in a fresh Google Colab CPU runtime on 2026-10-06 (see report for session log paths). All data was downloaded inside the Colab VM from the pinned Zenodo record 5579797 and GitHub commit `976974894aa11211656b5e97a4abb21bcc35b8fd`.

*日本語: 本ノートブックは検証カメラ arsmorris2 のアーカイブ済み予測に対し、著者の後処理パイプラインをそのまま実行し、手作業アノテーションで評価したものです。91,484枚の学習やVGG16推論自体は含みません。HMMデコードと評価指標は等価な移植実装です。*
